In [2]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
import matplotlib.markers as mmarkers
import seaborn as sns
import scipy

from compcats.localizer import localizer
from compcats.datasets.fmri import nsd
from compcats.utils import model_utils

import main_analysis as ma
import paths

# Matplotlib settings
import matplotlib as mpl
from matplotlib import rcParams
rcParams.update({'figure.autolayout': False})
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
# Font
font_path = '/home/alishd/.fonts/Avenir.ttc'
avenir_font = fm.FontProperties(fname=font_path)
mpl.rcParams['font.family'] = avenir_font.get_name()
ttc = fm.get_font(font_path)
print("Font name from TTC:", ttc.family_name)
print("Style:", ttc.style_name)

RESULTS_DIR = 'results'
MODELS_DIR = 'models'
PLOTS_DIR = f'{RESULTS_DIR}/plots/postvss25'
os.makedirs(PLOTS_DIR, exist_ok=True)

NSD_DIR = paths.NSD_DIR
GRILLSPECTOR_LOCALIZER_DIR = paths.GRILLSPECTOR_LOCALIZER_DIR
SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Font name from TTC: Avenir
Style: Book


# Intact vs. t-value=7 (Optimized alpha)

## Median

In [59]:
savefig = True

ridge_alphaopt_lesion_csv_path = f'{RESULTS_DIR}/nsd515_lesion_ems_ridge_roioptimizedalpha.csv'
lesion_csv = pd.read_csv(ridge_alphaopt_lesion_csv_path)
subjectsubject_correlations_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)

plot_dir = f'{PLOTS_DIR}/fig4/intact_t7_lesioned'
os.makedirs(plot_dir, exist_ok=True)

results_df = lesion_csv.copy()
mappers = ['ridge', 'l1positive']
subjects = [1, 2, 5, 7]
metrics = ['normed_pearsonr', 'normed_rdmr']

# Take median across subjects for each model, fmri_froi
results_df_froi_sbjmedian = results_df.groupby(
    ['model', 'fmri_froi']
).agg({
    'layer_raw_pearsonr': 'median',
    'layer_normed_pearsonr': 'median',
    'layer_raw_rdmr': 'median',
    'layer_normed_rdmr': 'median',
	'lesion_raw_pearsonr': 'median',
    'lesion_normed_pearsonr': 'median',
    'lesion_raw_rdmr': 'median',
    'lesion_normed_rdmr': 'median',
}).reset_index()
# Take median across frois for each model
results_df_froimedian = results_df_froi_sbjmedian.groupby(
    ['model']
).agg({
    'layer_raw_pearsonr': 'median',
    'layer_normed_pearsonr': 'median',
    'layer_raw_rdmr': 'median',
    'layer_normed_rdmr': 'median',
	'lesion_raw_pearsonr': 'median',
    'lesion_normed_pearsonr': 'median',
    'lesion_raw_rdmr': 'median',
    'lesion_normed_rdmr': 'median',
}).reset_index()

x_offset = 0.05
alpha = 0.5
markersize = 25
x = [0, 0.3]
# fig, axs = plt.subplots(len(metrics), len(mappers), figsize=(5*len(mappers), 5*len(metrics)), dpi=300)

for mpidx, mapper in enumerate(mappers):
    fig, axs = plt.subplots(len(metrics), 1, figsize=(5*1, 5*len(metrics)), dpi=300)
    for midx, metric in enumerate(metrics):
        if mapper == 'l1positive':
            print('L1-positive not implemented yet, making a mockup')
            froi_pts = lesion_pts = np.array([])
        else:
            froi_pts = np.array(results_df_froimedian[f'layer_{metric}'].tolist())
            lesion_pts = np.array(results_df_froimedian[f'lesion_{metric}'].tolist())
        x_froi = (np.ones(len(froi_pts)) * x[0]) - x_offset
        x_lesion = (np.ones(len(lesion_pts)) * x[1]) - x_offset
        axs[midx].scatter(
            x_froi, froi_pts, color='red', alpha=alpha, zorder=2,
            edgecolors='none', s=markersize
        )
        axs[midx].scatter(
            x_lesion, lesion_pts, color='red', alpha=alpha, zorder=2,
            edgecolors='none', s=markersize
        )
        # connect points with a line
        for i in range(len(froi_pts)):
            axs[midx].plot([x_froi[i], x_lesion[i]], [froi_pts[i], lesion_pts[i]], color='red', alpha=0.2, linewidth=0.5, zorder=1)

        # Bar plot with median and SEM
        bar_width = 0.2 #0.4
        medians = [np.median(froi_pts), np.median(lesion_pts)]
        sems = [froi_pts.std() / np.sqrt(len(froi_pts)), lesion_pts.std() / np.sqrt(len(lesion_pts))]

        axs[midx].bar(x, medians, yerr=sems, width=bar_width, color=['red', 'red'], alpha=0.25, capsize=0, error_kw={'linewidth': 1})
        # Conduct Welch's t-test
        t_value, p_value = scipy.stats.ttest_ind(froi_pts, lesion_pts, equal_var=False)
        # plt.text(0.95, 0.89,
        #          f"t-value: {t_value:.2f}\np-value: {p_value:.2e}",
        #          horizontalalignment='right', verticalalignment='bottom',
        #          transform=plt.gca().transAxes)
        print(f"t-value: {t_value:.2f}\np-value: {p_value:.2e}")

        axs[midx].axhline(y=0, color='black', linestyle='--', alpha=1, linewidth=1)

        axs[midx].spines['top'].set_visible(False)
        axs[midx].spines['right'].set_visible(False)
        axs[midx].spines['left'].set_position(('outward', 5))
        axs[midx].spines['bottom'].set_position(('outward', 5))
        axs[midx].set_yticks([0, 0.25, 0.5, 0.75, 1])
        axs[midx].spines['left'].set_bounds(0, 1)
        plt.ylim(0, 1)
        # plt.tick_params(axis='y', labelleft=False)
        axs[midx].spines['bottom'].set_bounds(x[0], x[1])
        axs[midx].set_xticks(x, ['Intact', 'Lesioned'])
        axs[midx].set_xlim(x[0]-0.25, x[1]+0.25)
        # plt.tick_params(axis='x', labelbottom=False)
        # plt.xlabel('t-threshold of model fROI lesioned', fontsize=12)
        axs[midx].set_ylabel(f'Noise-corrected {metric.split("_")[-1]} (median ± SEM)', fontsize=12)

        # if froi != 'FFA':
        #     plt.gca().spines['left'].set_visible(False)
        #     # Stop showing y-axis ticks
        #     plt.gca().yaxis.set_ticks([])

    plt.tight_layout()
    if savefig:
        plt.savefig(f'{plot_dir}/lesioned_{mapper}_medianfroi.pdf', dpi=300, bbox_inches='tight', transparent=True)
    else:
        plt.show()
    plt.close()

t-value: 0.42
p-value: 6.76e-01
t-value: 0.33
p-value: 7.45e-01
L1-positive not implemented yet, making a mockup
t-value: nan
p-value: nan
L1-positive not implemented yet, making a mockup
t-value: nan
p-value: nan


/om2/user/alishd/anaconda/envs/catsel/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/om2/user/alishd/anaconda/envs/catsel/lib/python3.12/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/om2/user/alishd/anaconda/envs/catsel/lib/python3.12/site-packages/numpy/_core/_methods.py:227: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/om2/user/alishd/anaconda/envs/catsel/lib/python3.12/site-packages/numpy/_core/_methods.py:184: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/om2/user/alishd/anaconda/envs/catsel/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/om2/user/alishd/ana

## Individual fROIs (Ridge)

In [ ]:
savefig = True

ridge_alphaopt_lesion_csv_path = f'{RESULTS_DIR}/nsd515_lesion_ems_ridge_roioptimizedalpha.csv'
lesion_csv = pd.read_csv(ridge_alphaopt_lesion_csv_path)
subjectsubject_correlations_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)

plot_dir = f'{PLOTS_DIR}/fig4/intact_t7_lesioned'
os.makedirs(plot_dir, exist_ok=True)

results_df = lesion_csv.copy()
frois = ['FFA', 'PPA', 'EBA']
subjects = [1, 2, 5, 7]
metrics = ['normed_pearsonr', 'normed_rdmr']

# Take median across subjects for each model, fmri_froi
results_df_froi_sbjmedian = results_df.groupby(
    ['model', 'fmri_froi']
).agg({
    'layer_raw_pearsonr': 'median',
    'layer_normed_pearsonr': 'median',
    'layer_raw_rdmr': 'median',
    'layer_normed_rdmr': 'median',
	'lesion_raw_pearsonr': 'median',
    'lesion_normed_pearsonr': 'median',
    'lesion_raw_rdmr': 'median',
    'lesion_normed_rdmr': 'median',
}).reset_index()

fig, axs = plt.subplots(len(metrics), len(frois), figsize=(5*len(frois), 5*len(metrics)), dpi=300)

x_offset = 0.05
alpha = 0.5
markersize = 25
x = [0, 0.3]

for fidx, froi in enumerate(frois):
    results_df_froi = results_df_froi_sbjmedian[results_df_froi_sbjmedian['fmri_froi'] == froi]
    for midx, metric in enumerate(metrics):
        froi_pts = np.array(results_df_froi[f'layer_{metric}'].tolist())
        lesion_pts = np.array(results_df_froi[f'lesion_{metric}'].tolist())

        x_froi = (np.ones(len(froi_pts)) * x[0]) - x_offset
        x_lesion = (np.ones(len(lesion_pts)) * x[1]) - x_offset
        axs[midx, fidx].scatter(
            x_froi, froi_pts, color='red', alpha=alpha, zorder=2,
            edgecolors='none', s=markersize
        )
        axs[midx, fidx].scatter(
            x_lesion, lesion_pts, color='red', alpha=alpha, zorder=2,
            edgecolors='none', s=markersize
        )
        # connect points with a line
        for i in range(len(froi_pts)):
            axs[midx, fidx].plot([x_froi[i], x_lesion[i]], [froi_pts[i], lesion_pts[i]], color='red', alpha=0.2, linewidth=0.5, zorder=1)

        # Bar plot with median and SEM
        bar_width = 0.2 #0.4
        medians = [np.median(froi_pts), np.median(lesion_pts)]
        sems = [froi_pts.std() / np.sqrt(len(froi_pts)), lesion_pts.std() / np.sqrt(len(lesion_pts))]

        axs[midx, fidx].bar(x, medians, yerr=sems, width=bar_width, color=['red', 'red'], alpha=0.25, capsize=0, error_kw={'linewidth': 1})
        # Conduct Welch's t-test
        t_value, p_value = scipy.stats.ttest_ind(froi_pts, lesion_pts, equal_var=False)
        print(f"froi: {froi}, metric: {metric}: t-value: {t_value:.2f}, p-value: {p_value:.2e}")

        # Label fROI
        axs[midx, fidx].text(0.9, 0.9,
                 froi,
                 horizontalalignment='right', verticalalignment='bottom',
                 transform=axs[midx, fidx].transAxes)

        axs[midx, fidx].axhline(y=0, color='black', linestyle='--', alpha=1, linewidth=1)

        axs[midx, fidx].spines['top'].set_visible(False)
        axs[midx, fidx].spines['right'].set_visible(False)
        axs[midx, fidx].spines['left'].set_position(('outward', 5))
        axs[midx, fidx].spines['bottom'].set_position(('outward', 5))
        axs[midx, fidx].set_yticks([0, 0.25, 0.5, 0.75, 1])
        axs[midx, fidx].spines['left'].set_bounds(0, 1)
        plt.ylim(0, 1)
        # plt.tick_params(axis='y', labelleft=False)
        axs[midx, fidx].spines['bottom'].set_bounds(x[0], x[1])
        axs[midx, fidx].set_xticks(x, ['Intact', 'Lesioned'])
        axs[midx, fidx].set_xlim(x[0]-0.25, x[1]+0.25)
        # plt.tick_params(axis='x', labelbottom=False)
        # plt.xlabel('t-threshold of model fROI lesioned', fontsize=12)
        axs[midx, fidx].set_ylabel(f'Noise-corrected {metric.split("_")[-1]} (median ± SEM)', fontsize=12)

        # if froi != 'FFA':
        #     plt.gca().spines['left'].set_visible(False)
        #     # Stop showing y-axis ticks
        #     plt.gca().yaxis.set_ticks([])

plt.tight_layout()
if savefig:
    plt.savefig(f'{plot_dir}/lesioned_ridge_individualfroi.pdf', dpi=300, bbox_inches='tight', transparent=True)
else:
    plt.show()
plt.close()

froi: FFA, metric: normed_pearsonr: t-value: 0.67, p-value: 5.06e-01
froi: FFA, metric: normed_rdmr: t-value: 0.78, p-value: 4.38e-01
froi: PPA, metric: normed_pearsonr: t-value: 0.10, p-value: 9.24e-01
froi: PPA, metric: normed_rdmr: t-value: 0.10, p-value: 9.17e-01
froi: EBA, metric: normed_pearsonr: t-value: 0.11, p-value: 9.13e-01
froi: EBA, metric: normed_rdmr: t-value: 0.15, p-value: 8.84e-01


## Individual fROIs (L1-positive)

In [ ]:
# TODO

# Across t-values (fixed alpha)

In [47]:
lesion_csv_path = f'{RESULTS_DIR}/nsd515_lesion_ems_fixedalpha.csv'
lesion_csv = pd.read_csv(lesion_csv_path)
# lesion_csvs = [f'{RESULTS_DIR}/nsd515_lesion_ems_fixedalpha_v{i}.csv' for i in range(1, 8)]

# if not os.path.exists(lesion_csv_path):
#     lesion_csv = pd.concat([pd.read_csv(lesion_csv) for lesion_csv in lesion_csvs])
#     lesion_csv.to_csv(lesion_csv_path, index=False)
# else:
#     lesion_csv = pd.read_csv(lesion_csv_path)
# print("Number of rows for each fROI:")
# print(lesion_csv['fmri_froi'].value_counts())

In [55]:
savefig = True

fmri_tthreshold = 7

plot_dir = f'{PLOTS_DIR}/fig4/across_tvals'
os.makedirs(plot_dir, exist_ok=True)

results_df = lesion_csv.copy()
subjectsubject_correlations_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
subjects = results_df['subject'].unique().tolist()

frois = ['FFA', 'PPA', 'EBA']
ordered_tthresholds = [-1, 11, 9, 7, 5, 3]
xticks_dict = {
    -1: 'Intact',
    11: '11',
    9: '9',
    7: '7',
    5: '5',
    3: '3'
}
metrics = ['normed_pearsonr', 'normed_rdmr']
plot_ceiling = False

x_offset = 0.05
alpha = 0.5
markersize = 25
x = [0, 0.3, 0.6, 0.9, 1.2, 1.5]
fig, axs = plt.subplots(len(metrics), len(frois), figsize=(5*len(frois), 5*len(metrics)), dpi=300)

for midx, metric in enumerate(metrics):
    for fidx, froi in enumerate(frois):
        print(metric, froi)
        # For specific froi & fmri_tthreshold, get results_df
        results_df_froi = results_df[
            (results_df['model_froi'] == froi) &
            (results_df['fmri_froi'] == froi) &
            (results_df['fmri_tthreshold'] == fmri_tthreshold)
        ]
        # Take median across subjects for each model, model_threshold
        results_df_froi_sbjmedian = results_df_froi.groupby(
            ['model', 'model_tthreshold']
        ).agg({
            'raw_pearsonr': 'median',
            'normed_pearsonr': 'median',
            'raw_rdmr': 'median',
            'normed_rdmr': 'median',
            'n_units_layer': 'mean',
            'n_units_lesioned': 'mean',
            'percent_units_lesioned': 'mean',
        }).reset_index()

        if plot_ceiling:
            ceiling_values = []
            for subject1_idx in range(len(subjects)):
                for subject2_idx in range(subject1_idx, len(subjects)):
                    subject1 = subjects[subject1_idx]
                    subject2 = subjects[subject2_idx]
                    if subject1 == subject2:
                        continue
                    subjectsubject_froi = subjectsubject_correlations_csv[
                        (subjectsubject_correlations_csv['fmri_froi'] == froi) &
                        (subjectsubject_correlations_csv['subject1'] == subject1) &
                        (subjectsubject_correlations_csv['subject2'] == subject2) & 
                        (subjectsubject_correlations_csv['fmri_tthreshold'] == 7)
                    ]
                    ceiling_metric_ = 'pearsonr' if metric == 'raw_pearsonr' else metric
                    ceiling_values += list(subjectsubject_froi[ceiling_metric_])
                    # print(f'Subject {subject1} vs Subject {subject2}: {subjectsubject_froi[ceiling_metric]}')
            ceiling_values = np.array(ceiling_values).flatten()
            # print(f'Ceiling values: {ceiling_values}')
            ceiling_median = np.median(ceiling_values)
            ceiling_sem = np.std(ceiling_values) / np.sqrt(len(ceiling_values))

            # Plot horizontal line at median of ceiling values
            plt.axhline(y=ceiling_median, color='royalblue', linestyle='-', alpha=0.8, linewidth=1, zorder=1)

            # Fill area around the line for SEM
            plt.fill_between(
                [x[0]-0.5, x[1]+0.5],  # Extend across full x range
                ceiling_median - ceiling_sem,
                ceiling_median + ceiling_sem,
                color='royalblue', alpha=0.2, zorder=0, edgecolor='none'
            )

        
        # model_tthresholds = results_df_froi_sbjmedian['model_tthreshold'].unique()
        model_tthresholds = ordered_tthresholds
        assert set(model_tthresholds) == set(ordered_tthresholds), \
            f"Model tthresholds {model_tthresholds} are same as ordered tthresholds {ordered_tthresholds}"
        n_model_tthresholds = len(model_tthresholds)

        # percent_lesioned_means = []
        # percent_lesioned_sems = []

        for tidx, model_tthreshold in enumerate(model_tthresholds):
            tthreshold_results = results_df_froi_sbjmedian[
                (results_df_froi_sbjmedian['model_tthreshold'] == model_tthreshold)
            ]
            y_pts = tthreshold_results[metric].values
            percent_lesioneds = tthreshold_results['percent_units_lesioned'].values
            pl_mean = np.mean(percent_lesioneds)
            pl_sem = np.std(percent_lesioneds) / np.sqrt(len(percent_lesioneds))
            # percent_lesioned_means.append(np.mean(percent_lesioneds))
            # percent_lesioned_sems.append(np.std(percent_lesioneds) / np.sqrt(len(percent_lesioneds)))
            # Add text on top of each x point mentioning the percent lesioned mean +- sem
            axs[midx, fidx].text(
                x[tidx] + x_offset, 0.95,
                f"{pl_mean:.1f} ± {pl_sem:.1f}%",
                horizontalalignment='center', verticalalignment='bottom',
                fontsize=8, color='black', alpha=0.8
            )
        
            x_froi = (np.ones(len(y_pts)) * x[tidx]) - x_offset
            axs[midx, fidx].scatter(
                x_froi, y_pts, color='red', alpha=alpha, zorder=2,
                edgecolors='none', s=markersize
            )
            # connect points with a line
            # for i in range(len(froi_pts)):
            #     axs[midx, fidx].plot([x_froi[i], x_lesion[i]], [froi_pts[i], lesion_pts[i]], color='red', alpha=0.2, linewidth=0.5, zorder=1)

            # Bar plot with median and SEM
            bar_width = 0.2 #0.4
            median = np.median(y_pts)
            sem = y_pts.std() / np.sqrt(len(y_pts))

            axs[midx, fidx].bar(x[tidx], median, yerr=sem, width=bar_width, color='red', alpha=0.25, capsize=0, error_kw={'linewidth': 1})
            # # Conduct Welch's t-test
            # t_value, p_value = scipy.stats.ttest_ind(froi_pts, lesion_pts, equal_var=False)
            # # axs[midx, fidx].text(0.95, 0.89,
            # #          f"t-value: {t_value:.2f}\np-value: {p_value:.2e}",
            # #          horizontalalignment='right', verticalalignment='bottom',
            # #          transform=axs[midx, fidx].transAxes)
            # print(f"t-value: {t_value:.2f}\np-value: {p_value:.2e}")
        axs[midx, fidx].text(0.95, 0.89,
                    froi,
                    horizontalalignment='right', verticalalignment='bottom',
                    transform=axs[midx, fidx].transAxes)

        axs[midx, fidx].axhline(y=0, color='black', linestyle='--', alpha=1, linewidth=1)

        axs[midx, fidx].spines['top'].set_visible(False)
        axs[midx, fidx].spines['right'].set_visible(False)
        axs[midx, fidx].spines['left'].set_position(('outward', 5))
        axs[midx, fidx].spines['bottom'].set_position(('outward', 5))
        axs[midx, fidx].set_yticks([0, 0.25, 0.5, 0.75, 1])
        axs[midx, fidx].spines['left'].set_bounds(0, 1)
        axs[midx, fidx].set_ylim(0, 1)
        # axs[midx, fidx].tick_params(axis='y', labelleft=False)
        axs[midx, fidx].spines['bottom'].set_bounds(x[0], x[-1])
        xticklabels = [xticks_dict[t] for t in model_tthresholds]
        axs[midx, fidx].set_xticks(x, xticklabels, fontsize=10)
        # axs[midx, fidx].xticks(x, model_tthresholds)
        axs[midx, fidx].set_xlim(x[0]-0.25, x[-1]+0.25)
        # axs[midx, fidx].tick_params(axis='x', labelbottom=False)
        # axs[midx, fidx].xlabel('t-threshold of model fROI lesioned', fontsize=12)
        axs[midx, fidx].set_ylabel(f'Noise-corrected {metric.split("_")[-1]} (median ± SEM)', fontsize=12)

        # if froi != 'FFA':
        #     axs[midx, fidx].spines['left'].set_visible(False)
        #     # Stop showing y-axis ticks
        #     axs[midx, fidx].yaxis.set_ticks([])
        # plt.title(f'{froi} fROI, mapper: Ridge', fontsize=14, pad=10)

        # Stats
        # Run an ANOVA on normed_pearsonr across model_tthresholds to see if there is a significant difference
        anova_results = scipy.stats.f_oneway(
            *[results_df_froi_sbjmedian[results_df_froi_sbjmedian['model_tthreshold'] == t][metric].values for t in model_tthresholds]
        )
        print(f"ANOVA results for {froi}, metric {metric}: F-value = {anova_results.statistic:.3f}, p-value = {anova_results.pvalue:.3e}")
        # Print the median and SEM for each model_tthreshold
        # for model_tthreshold in model_tthresholds:
        #     tthreshold_results = results_df_froi_sbjmedian[
        #         (results_df_froi_sbjmedian['model_tthreshold'] == model_tthreshold)
        #     ]
        #     median = tthreshold_results[metric].median()
        #     sem = tthreshold_results[metric].std() / np.sqrt(len(tthreshold_results))
            # print(f"Model tthreshold {model_tthreshold}: median = {median:.3f}, SEM = {sem:.3f}")
        # Run t-tests between neighboring pairs of ordered_tthresholds
        for i in range(len(ordered_tthresholds) - 1):
            t1 = ordered_tthresholds[0]
            t2 = ordered_tthresholds[i + 1]
            t1_results = results_df_froi_sbjmedian[results_df_froi_sbjmedian['model_tthreshold'] == t1][metric].values
            t2_results = results_df_froi_sbjmedian[results_df_froi_sbjmedian['model_tthreshold'] == t2][metric].values
            t_stat, p_val = scipy.stats.ttest_ind(t1_results, t2_results, equal_var=False)
            print(f"T-test between {t1} and {t2}: t-statistic = {t_stat:.3f}, p-value = {p_val:.3e}")

plt.tight_layout()
if savefig:
    plt.savefig(f'{plot_dir}/lesioned_tvals_ridge_individualfroi.pdf', dpi=300, bbox_inches='tight', transparent=True)
else:
    plt.show()
plt.close()

normed_pearsonr FFA
ANOVA results for FFA, metric normed_pearsonr: F-value = 0.129, p-value = 9.858e-01
T-test between -1 and 11: t-statistic = 0.331, p-value = 7.416e-01
T-test between -1 and 9: t-statistic = 0.421, p-value = 6.747e-01
T-test between -1 and 7: t-statistic = 0.518, p-value = 6.058e-01
T-test between -1 and 5: t-statistic = 0.615, p-value = 5.403e-01
T-test between -1 and 3: t-statistic = 0.726, p-value = 4.700e-01
normed_pearsonr PPA
ANOVA results for PPA, metric normed_pearsonr: F-value = 0.008, p-value = 1.000e+00
T-test between -1 and 11: t-statistic = 0.055, p-value = 9.561e-01
T-test between -1 and 9: t-statistic = 0.066, p-value = 9.477e-01
T-test between -1 and 7: t-statistic = 0.094, p-value = 9.251e-01
T-test between -1 and 5: t-statistic = 0.125, p-value = 9.011e-01
T-test between -1 and 3: t-statistic = 0.189, p-value = 8.507e-01
normed_pearsonr EBA
ANOVA results for EBA, metric normed_pearsonr: F-value = 0.009, p-value = 1.000e+00
T-test between -1 and 11: 